# Ajuste fino con cortes sin pelvis (negativos)

**Problema.** El modelo multitarea se entrenó solo con cortes que tienen pelvis (`manifest_cortes_positivos.csv`). Al recorrer un volumen completo, los cortes de arriba y de abajo, donde hay fémures o columna, reciben cajas con score alto. Con el modelo actual (entrada 2.5D), en `val` 229 de 394 cortes sin pelvis tienen alguna caja y, con `umbral_corte = 0,08`, pasan 212 (54 %). El modelo anterior, de un solo corte, dejaba pasar 190 (48 %) con 0,45: el cambio a 2.5D no resolvió el problema. Ninguna métrica actual lo ve, porque la mAP y el Dice se calculan solo sobre cortes positivos.

**Qué hace este notebook.** Parte del modelo actual (`modelo_multitarea_final.pt`) y lo ajusta unas pocas épocas mezclando los cortes positivos de siempre con cortes **sin un solo vóxel de pelvis**. En un negativo el target es "nada": sin cajas, semántica toda de fondo y sin centros. Se comparan dos proporciones de negativos y se elige con `val`. `test` no se usa.

**Regla de selección, escrita antes de entrenar** (`src/cortes_vacios.CRITERIO`):
1. Se mide la línea base con el modelo actual, con este mismo código.
2. Una época **cumple** si, frente a la línea base, la mAP@0.50 en cortes positivos no baja más de 0,01, el Dice con cajas predichas no baja más de 0,01 y, con el umbral recalibrado, se conserva al menos el 99 % de los cortes con pelvis.
3. Entre las épocas que cumplen, gana la que deja pasar menos cortes vacíos (`fpr_vacios_cal`).
4. Si ninguna cumple, **se conserva el modelo actual** y el resultado se reporta como negativo.

**Cómo correrlo**
1. `SMOKE_TEST = True` (celda 1) y *Run All*: unos minutos. Comprueba que todo conecta, sin tocar los resultados oficiales.
2. `SMOKE_TEST = False` y *Run All*: la corrida real. Lo más largo son los cachés de negativos (solo la primera vez) y dos ajustes de 4 épocas.
3. Revisar la sección 8. Solo si el ganador cumple, poner `ADOPTAR = True` en la sección 10 y ejecutar esa celda.

**Salidas:** `resultados/datos/targets/manifest_cortes_negativos.csv`, `resultados/metricas/entrenamiento_negativos/`, `resultados/figuras/entrenamiento_negativos/` y `resultados/modelos/entrenamiento_negativos/`.

## 1. Configuración

Los hiperparámetros se leen del `config.json` del entrenamiento original para que el ajuste use exactamente los mismos (batch, pesos de pérdida, jitter, recorte de gradiente). Lo único nuevo es la etapa de ajuste: 4 épocas, tasa de aprendizaje 1e-4 (un tercio de la etapa C) y los negativos.

El dispositivo se elige en este orden: CUDA, Apple MPS y CPU. La precisión mixta (AMP) solo se usa en CUDA, igual que en el entrenamiento original; si se corre en otro dispositivo, `config.json` lo deja registrado.

In [ ]:
import ast, gc, hashlib, json, math, random, shutil, subprocess, sys, time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy.ndimage import maximum_filter
from skimage.exposure import equalize_adapthist
from skimage.filters import threshold_otsu

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements.txt").is_file())
BASE_DIR = ROOT                                   # lo usan resolve_repo_path y las definiciones del equipo
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.rutas import CACHE_DIR, DATASET_CSV, DATOS_DIR, FIGURAS_DIR, MANIFEST_CSV, METRICAS_DIR, MODELOS_DIR, NOTEBOOKS_DIR
from src.definiciones_notebooks import cargar_definiciones
from src import cortes_vacios as cv

SMOKE_TEST = True          # 1) True: prueba corta que no toca los resultados oficiales. 2) False: corrida real.
SEED = 42

# ---------------------------------------------------------------- modelo de partida
ORIGEN_DIR = MODELOS_DIR / "entrenamiento_multitarea"
RESPALDO_SIN_NEGATIVOS = ORIGEN_DIR / "modelo_multitarea_final_sin_negativos.pt"
# Si ya se adoptó un modelo ajustado (sección 10), el punto de partida sigue siendo el modelo SIN negativos.
MODELO_INICIAL = RESPALDO_SIN_NEGATIVOS if RESPALDO_SIN_NEGATIVOS.is_file() else ORIGEN_DIR / "modelo_multitarea_final.pt"
if not MODELO_INICIAL.is_file():
    raise FileNotFoundError(f"No encuentro {MODELO_INICIAL}")
if MODELO_INICIAL.stat().st_size < 10_000:
    raise RuntimeError(f"{MODELO_INICIAL.name} es un puntero de Git LFS: ejecutar `git lfs install` y `git lfs pull`.")

# ---------------------------------------------------------------- mismos hiperparámetros del entrenamiento original
CONFIG_ORIGINAL = json.loads((METRICAS_DIR / "entrenamiento_multitarea" / "config.json").read_text(encoding="utf-8"))
IMAGE_SIZE = CONFIG_ORIGINAL["IMAGE_SIZE"]
GRID_SIZE = CONFIG_ORIGINAL["GRID_SIZE"]
BACKBONE_NAME = CONFIG_ORIGINAL["BACKBONE_NAME"]
USE_CBAM = CONFIG_ORIGINAL["USE_CBAM"]
USE_CBAM_GATE = CONFIG_ORIGINAL["USE_CBAM_GATE"]
USE_SKIPS = CONFIG_ORIGINAL["USE_SKIPS"]
BATCH_SIZE = CONFIG_ORIGINAL["BATCH_SIZE"]
VAL_BATCH_SIZE = CONFIG_ORIGINAL["VAL_BATCH_SIZE"]
WEIGHT_DECAY = CONFIG_ORIGINAL["WEIGHT_DECAY"]
GRAD_CLIP = CONFIG_ORIGINAL["GRAD_CLIP"]
SEG_WEIGHTS = tuple(CONFIG_ORIGINAL["SEG_WEIGHTS"])
JITTER_PX = CONFIG_ORIGINAL["JITTER_PX"]
PRED_BOX_SCORE_MIN = CONFIG_ORIGINAL["PRED_BOX_SCORE_MIN"]
INSTANCE_EVAL_N = CONFIG_ORIGINAL["INSTANCE_EVAL_N"]
EVAL_SCORE_MIN = CONFIG_ORIGINAL["EVAL_SCORE_MIN"]
IN_CHANNELS = CONFIG_ORIGINAL.get("IN_CHANNELS", 1)          # 3 = entrada 2.5D (cortes z-1, z, z+1); los modelos anteriores no lo guardaban
CENTER_MODE = CONFIG_ORIGINAL.get("CENTER_MODE", "mean")     # centro objetivo del decoder con el que se entrenó el modelo
NUM_WORKERS = 0
LOG_GRAD_EVERY = 50
MAX_TRAIN_SLICES = MAX_VAL_SLICES = MAX_NEG_SLICES = None

# ---------------------------------------------------------------- el experimento
ETAPA_AJUSTE = {"epochs": 4, "lr": 1e-4, "train": ["backbone", "cbam", "det_head", "seg_decoder"],
                "w_det": 1.0, "w_seg": 1.0, "prior": "mixed", "p_pred_end": 0.5, "select": "negativos"}
CONFIGURACIONES = [                                 # fracción de los cortes negativos de train que entra al ajuste
    {"nombre": "todos", "fraccion": 1.0},           # ~5.700 negativos: la proporción natural de un volumen (~1 por cada 3 positivos)
    {"nombre": "mitad", "fraccion": 0.5},           # la mitad, repartida por caso: ~1 por cada 6 positivos
]
RECALL_MIN = cv.CRITERIO["recall_min"]
UMBRAL_ACTUAL = json.loads((METRICAS_DIR / "nms_pertenencia" / "umbral_corte.json").read_text(encoding="utf-8"))["umbral_corte"]

if SMOKE_TEST:
    MAX_TRAIN_SLICES, MAX_VAL_SLICES, MAX_NEG_SLICES, INSTANCE_EVAL_N = 64, 32, 32, 8
    BATCH_SIZE = VAL_BATCH_SIZE = 8
    ETAPA_AJUSTE["epochs"] = 1

ETAPA = "entrenamiento_negativos_smoke" if SMOKE_TEST else "entrenamiento_negativos"
OUT_ROOT, MODEL_ROOT, FIG_DIR = METRICAS_DIR / ETAPA, MODELOS_DIR / ETAPA, FIGURAS_DIR / ETAPA
for _d in (OUT_ROOT, MODEL_ROOT, FIG_DIR, CACHE_DIR):
    _d.mkdir(parents=True, exist_ok=True)
OUT_DIR, MODEL_DIR = OUT_ROOT, MODEL_ROOT          # train_stage escribe aquí; se cambian por configuración (sección 7)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
USE_AMP = DEVICE.type == "cuda"
NOMBRE_DISPOSITIVO = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else DEVICE.type

print(f"PyTorch {torch.__version__} | dispositivo: {NOMBRE_DISPOSITIVO} | AMP: {USE_AMP} | SMOKE_TEST: {SMOKE_TEST}")
print("Modelo de partida:", MODELO_INICIAL.name, "| canales de entrada:", IN_CHANNELS, "| centros:", CENTER_MODE,
      "| umbral_corte actual:", UMBRAL_ACTUAL)
print("Salidas en:", OUT_ROOT)
if not USE_AMP:
    print("AVISO: sin precisión mixta (solo existe en CUDA). Queda registrado en config.json.")

## 2. Cargar las definiciones del equipo

Se reutiliza el código existente sin copiarlo: datos y targets, backbone, CBAM, decoder y, del notebook `Entrenamiento_multitarea`, el modelo, el caché, las pérdidas, la evaluación y el bucle `train_stage`. Si alguna definición cambia de nombre, la carga se detiene con un error.

In [ ]:
n_def = 0
n_def += cargar_definiciones(NOTEBOOKS_DIR / "Dataset_splits.ipynb", {"resolve_repo_path"}, destino=globals())
n_def += cargar_definiciones(NOTEBOOKS_DIR / "Ventaneo_hu.ipynb", {"apply_hu_window", "enhance_bone_contrast"}, destino=globals())
n_def += cargar_definiciones(NOTEBOOKS_DIR / "Targets_deteccion.ipynb",
                             {"instance_to_semantic", "boxes_from_semantic", "letterbox_pair", "build_detection_target",
                              "PengwinDetectionDataset"}, valores={"CLASS_ID_TO_NAME"}, destino=globals())
n_def += cargar_definiciones(NOTEBOOKS_DIR / "Backbone.ipynb", {"FundidoraPC", "FundidoraPCPlus", "build_backbone"}, destino=globals())
n_def += cargar_definiciones(NOTEBOOKS_DIR / "Segmentacion_instancias.ipynb",
                             {"build_instance_segmentation_targets", "build_detection_region_priors", "SegmentationConvBlock",
                              "backbone_feature_pyramid", "SkipFusionBlock", "CenterOffsetSkipDecoder",
                              "center_offset_segmentation_loss", "soft_dice_semantic_loss",
                              "decode_center_offset_instances"}, destino=globals())
n_def += cargar_definiciones(NOTEBOOKS_DIR / "cbam.ipynb", {"ChannelAttention", "SpatialAttention", "CBAM", "CompuertaResidualCBAM"},
                             destino=globals())
n_def += cargar_definiciones(
    NOTEBOOKS_DIR / "Entrenamiento_multitarea.ipynb",
    {"seed_everything", "autocast", "_sha1", "_VolumeSlices", "LeanPengwinDetectionDataset", "build_or_load_cache", "_read_cache",
     "pad_boxes", "unpad_boxes", "CachedSliceDataset", "collate_slices", "MultiTaskNet", "jitter_boxes", "predicted_boxes_top1",
     "make_priors", "make_grid_targets", "seg_loss_from_batch", "compute_losses", "_to_np", "_iou_np", "_average_precision",
     "detection_metrics", "fragment_hits", "evaluate", "selection_score", "set_trainable", "set_train_mode", "group_grad_norms",
     "trunk_grad_balance", "save_checkpoint", "load_weights", "train_stage", "export_inference_checkpoint", "load_trained_model"},
    destino=globals())

from src.deteccion_grid import (CH_BOX, CH_CLS, CH_OBJ, CLASS_NAMES, GRID_S, IMG_SIZE, LAMBDAS, STRIDE, box_iou, build_targets,
                                decode_boxes, decode_predictions, encode_boxes, grid_detection_loss, GridDetectionHead,
                                nms_por_clase, pairwise_iou, postprocess, split_head_output)
from src.pipeline_pengwin import CONFIG_PIPELINE, scores_por_corte
from src.entrada_25d import apilar_cortes, in_channels_del_modelo

assert GRID_S == GRID_SIZE and IMG_SIZE == IMAGE_SIZE, "la cabeza grid espera otra resolución"
DET_LAMBDAS = dict(LAMBDAS)
assert DET_LAMBDAS == CONFIG_ORIGINAL["DET_LAMBDAS"], "los λ de detección no son los del entrenamiento original"
print(f"Definiciones cargadas: {n_def} | λ detección: {DET_LAMBDAS}")

# Tablas globales que usa PengwinDetectionDataset
df = pd.read_csv(DATASET_CSV, dtype={"case_id": str})
slice_manifest = pd.read_csv(MANIFEST_CSV, dtype={"case_id": str})
DATA_VERSION = {"dataset_csv": _sha1(DATASET_CSV), "manifest": _sha1(MANIFEST_CSV), "image_size": IMAGE_SIZE, "channels": IN_CHANNELS}
if DATA_VERSION != CONFIG_ORIGINAL["DATA_VERSION"]:
    # En Windows Git convierte los CSV a CRLF y la huella cambia aunque los datos sean los mismos: el caché se reconstruye.
    print("AVISO: la huella de dataset.csv / manifiesto no coincide con la del entrenamiento original:",
          DATA_VERSION, "vs", CONFIG_ORIGINAL["DATA_VERSION"], "-> el caché de positivos se reconstruye.")

## 3. Manifiesto de cortes negativos

Un corte es negativo si su label no tiene **ni un vóxel** de pelvis. Los cortes con muy poca pelvis, que tampoco están en el manifiesto de positivos (menos de 20 píxeles), quedan fuera de los dos: son ambiguos y no se usan. Para cada negativo se guarda su distancia a la pelvis, solo como descripción. `posicion` se refiere al índice z del array (`antes` y `despues` del tramo con pelvis), sin suponer qué lado es anatómicamente arriba.

In [ ]:
NEG_CSV = DATOS_DIR / "targets" / "manifest_cortes_negativos.csv"


def leer_label(case_id):
    fila = df[df["case_id"] == case_id].iloc[0]
    return sitk.GetArrayFromImage(sitk.ReadImage(str(resolve_repo_path(fila["label_path"]))))


if NEG_CSV.is_file():
    negativos = pd.read_csv(NEG_CSV, dtype={"case_id": str})
    print("Manifiesto de negativos cargado:", NEG_CSV.relative_to(ROOT))
else:
    t0 = time.time()
    negativos = cv.manifiesto_negativos(df[["case_id", "split"]], leer_label)
    negativos.to_csv(NEG_CSV, index=False)
    print(f"Manifiesto de negativos creado en {time.time() - t0:.0f} s:", NEG_CSV.relative_to(ROOT))

# Ningún corte puede ser positivo y negativo a la vez
cruce = negativos.merge(slice_manifest, on=["case_id", "slice_index"], how="inner")
assert cruce.empty, f"{len(cruce)} cortes están en los dos manifiestos"
# Los splits del manifiesto coinciden con dataset.csv (sin fuga de pacientes)
assert (negativos.merge(df[["case_id", "split"]], on="case_id", suffixes=("", "_ds"))
        .eval("split == split_ds").all()), "el split de algún negativo no coincide con dataset.csv"

resumen_neg = pd.DataFrame({
    "positivos": slice_manifest.groupby("split").size(),
    "negativos": negativos.groupby("split").size(),
}).fillna(0).astype(int)
resumen_neg["negativos_por_positivo"] = (resumen_neg["negativos"] / resumen_neg["positivos"]).round(3)
display(resumen_neg)
print("Posición de los negativos:", negativos["posicion"].value_counts().to_dict())
print("Distancia a la pelvis (cortes):", negativos["distancia_a_pelvis"].describe()[["min", "50%", "max"]].to_dict())

## 4. Cachés de cortes

Los positivos reutilizan el mismo caché del entrenamiento original (`resultados/cache/train` y `val`), así que no se recalculan si la huella coincide. Los negativos se guardan aparte (`resultados/cache/neg_train` y `neg_val`) con el mismo preprocesamiento: ventana HU, Otsu + CLAHE y letterbox a 256 × 256. Construir los negativos de train es lo más lento y solo pasa una vez.

In [ ]:
def build_or_load_neg_cache(split, manifiesto, max_slices=None):
    # Igual que build_or_load_cache (Entrenamiento_multitarea), pero con los cortes del manifiesto de negativos.
    recs = manifiesto[manifiesto["split"] == split].sort_values(["case_id", "slice_index"])[["case_id", "slice_index"]]
    if max_slices:
        recs = recs.head(max_slices)
    tag = f"neg_{split}" + (f"_first{max_slices}" if max_slices else "") + ("_25d" if IN_CHANNELS == 3 else "")
    folder = CACHE_DIR / tag
    key = {**DATA_VERSION, "negativos": hashlib.sha1(recs.to_csv(index=False).encode()).hexdigest()[:12],
           "max_slices": max_slices}
    key_path = folder / "key.json"
    if key_path.exists() and json.loads(key_path.read_text()) == key:
        return _read_cache(tag, folder)

    folder.mkdir(parents=True, exist_ok=True)
    ds = LeanPengwinDetectionDataset(split=split, image_size=IMAGE_SIZE, channels=1, as_torch=False,
                                     slices_25d=IN_CHANNELS == 3)      # misma regla de vecinos que el caché de positivos
    ds.records = [{"case_id": str(c), "slice_index": int(s)} for c, s in zip(recs["case_id"], recs["slice_index"])]
    n, S = len(ds.records), IMAGE_SIZE
    images = np.lib.format.open_memmap(folder / "images.npy", mode="w+", dtype=np.float16, shape=(n, IN_CHANNELS, S, S))
    inst = np.lib.format.open_memmap(folder / "inst.npy", mode="w+", dtype=np.uint8, shape=(n, S, S))
    boxes_list, labels_list, case_ids, slice_ids = [], [], [], []
    t0 = time.perf_counter()
    for k in range(n):
        image, target = ds[k]
        mask = np.asarray(target["instance_mask"])
        assert np.asarray(image).shape == (IN_CHANNELS, S, S), np.asarray(image).shape
        assert len(target["boxes"]) == 0 and mask.max() == 0, f"el corte negativo {ds.records[k]} tiene pelvis"
        images[k] = np.asarray(image, dtype=np.float32).astype(np.float16)
        inst[k] = mask.astype(np.uint8)
        boxes_list.append(np.zeros((0, 4), np.float32)); labels_list.append(np.zeros(0, np.int64))
        case_ids.append(str(target["case_id"])); slice_ids.append(int(target["slice_index"]))
        if (k + 1) % 1000 == 0 or k + 1 == n:
            print(f"[{tag}] {k + 1}/{n} cortes | {time.perf_counter() - t0:.0f} s")
    boxes, labels = pad_boxes(boxes_list, labels_list)
    images.flush(); inst.flush()
    del images, inst, ds
    gc.collect()
    np.save(folder / "boxes.npy", boxes); np.save(folder / "labels.npy", labels)
    pd.DataFrame({"case_id": case_ids, "slice_index": slice_ids}).to_csv(folder / "meta.csv", index=False)
    key_path.write_text(json.dumps(key))          # al final: una construcción interrumpida no queda como válida
    print(f"[{tag}] caché creada: {n} cortes ({folder})")
    return _read_cache(tag, folder, quiet=True)


train_cache = build_or_load_cache("train", MAX_TRAIN_SLICES)
val_cache = build_or_load_cache("val", MAX_VAL_SLICES)
neg_train_cache = build_or_load_neg_cache("train", negativos, MAX_NEG_SLICES)
neg_val_cache = build_or_load_neg_cache("val", negativos, MAX_NEG_SLICES)

casos = {k: set(c["meta"]["case_id"]) for k, c in
         {"train": train_cache, "val": val_cache, "neg_train": neg_train_cache, "neg_val": neg_val_cache}.items()}
assert not ((casos["train"] | casos["neg_train"]) & (casos["val"] | casos["neg_val"])), "hay casos de val en train"
assert (neg_train_cache["labels"] == 0).all() and (neg_val_cache["labels"] == 0).all(), "un negativo trae cajas"
print({k: len(c["images"]) for k, c in {"train": train_cache, "val": val_cache, "neg_train": neg_train_cache,
                                         "neg_val": neg_val_cache}.items()})

## 5. Datos de entrenamiento y validación

- **Entrenamiento:** los positivos de siempre más una fracción de los negativos de train, mezclados en cada época.
- **Validación de detección y segmentación:** solo los cortes positivos de `val`, igual que en el entrenamiento original. Así la mAP y el Dice se comparan con la misma vara.
- **Validación de cortes vacíos:** el mejor score después del NMS en todos los cortes de `val`, positivos y negativos. Con eso se recalibra el umbral (recall de pelvis ≥ 0,99) y se mide cuántos cortes vacíos pasan.

In [ ]:
class MezclaCortes(torch.utils.data.Dataset):
    # Positivos + un subconjunto de negativos. `need_seg` se propaga a los dos (train_stage lo cambia por etapa).
    def __init__(self, positivos, negativos, indices_negativos):
        self.pos, self.neg = positivos, negativos
        self.idx_neg = [int(i) for i in indices_negativos]

    @property
    def need_seg(self):
        return self.pos.need_seg

    @need_seg.setter
    def need_seg(self, value):
        self.pos.need_seg = self.neg.need_seg = bool(value)

    def __len__(self):
        return len(self.pos) + len(self.idx_neg)

    def __getitem__(self, i):
        return self.pos[i] if i < len(self.pos) else self.neg[self.idx_neg[i - len(self.pos)]]


INDICE_NEG_TRAIN = {(str(c), int(s)): k for k, (c, s) in
                    enumerate(zip(neg_train_cache["meta"]["case_id"], neg_train_cache["meta"]["slice_index"]))}
negativos_train_en_cache = negativos[[(str(c), int(s)) in INDICE_NEG_TRAIN
                                      for c, s in zip(negativos["case_id"], negativos["slice_index"])]]


def loader_entrenamiento(fraccion):
    sel = cv.submuestrear_negativos(negativos_train_en_cache, fraccion, seed=SEED)
    idx = [INDICE_NEG_TRAIN[(str(c), int(s))] for c, s in zip(sel["case_id"], sel["slice_index"])]
    ds = MezclaCortes(CachedSliceDataset(train_cache, CENTER_MODE), CachedSliceDataset(neg_train_cache, CENTER_MODE), idx)
    gen = torch.Generator(); gen.manual_seed(SEED)
    loader = torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS,
                                         pin_memory=DEVICE.type == "cuda", collate_fn=collate_slices, generator=gen)
    return loader, len(idx)


val_loader = torch.utils.data.DataLoader(CachedSliceDataset(val_cache, CENTER_MODE), batch_size=VAL_BATCH_SIZE, shuffle=False,
                                         num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda", collate_fn=collate_slices)

# Comprobación: un batch mezclado con negativos tiene la forma esperada y sus negativos no traen cajas ni máscara
_l, _n = loader_entrenamiento(1.0)
_ds = _l.dataset
_item_neg = _ds[len(_ds.pos)]
assert len(_item_neg["boxes"]) == 0 and int(_item_neg["instance_mask"].max()) == 0 and int(_item_neg["semantic"].max()) == 0
assert float(_item_neg["centers"].max()) == 0.0 and not bool(_item_neg["foreground"].any())
_b = collate_slices([_ds[0], _item_neg])
print(f"train: {len(_ds.pos)} positivos + {_n} negativos | batch: imagen {tuple(_b['image'].shape)}, "
      f"semántica {tuple(_b['semantic'].shape)} | val positivos: {len(val_loader.dataset)} | val negativos: {len(neg_val_cache['images'])}")
del _l, _ds, _item_neg, _b

## 6. Modelo de partida, métricas de cortes vacíos y línea base

`evaluate` (del entrenamiento original) se envuelve para que, además de la mAP y el Dice en cortes positivos, devuelva las métricas de cortes vacíos de `src/cortes_vacios.py`. `selection_score` se extiende con el criterio `"negativos"`. `train_stage` usa estas dos funciones sin saberlo, así que el bucle de entrenamiento es literalmente el mismo de la etapa C.

La línea base se mide con el modelo actual y con este mismo código. Como comprobación, su mAP y su Dice deben coincidir (salvo variaciones pequeñas) con los guardados en el checkpoint original.

In [ ]:
PAYLOAD_INICIAL = torch.load(MODELO_INICIAL, map_location="cpu", weights_only=True)
_arq = PAYLOAD_INICIAL["architecture"]
assert (_arq["backbone_name"], _arq["use_cbam"], _arq.get("use_gate", True), _arq.get("use_skips", True)) == \
       (BACKBONE_NAME, USE_CBAM, USE_CBAM_GATE, USE_SKIPS), f"la arquitectura del checkpoint no coincide: {_arq}"
assert in_channels_del_modelo(PAYLOAD_INICIAL) == IN_CHANNELS, \
    f"el modelo de partida tiene {in_channels_del_modelo(PAYLOAD_INICIAL)} canales y config.json dice {IN_CHANNELS}"
ESTADO_INICIAL = PAYLOAD_INICIAL["model_state_dict"]


def modelo_inicial():
    seed_everything(SEED)
    m = MultiTaskNet(BACKBONE_NAME, USE_CBAM, USE_CBAM_GATE, USE_SKIPS, IN_CHANNELS).to(DEVICE)
    m.load_state_dict(ESTADO_INICIAL)        # estricto: falla si sobra o falta una clave
    return m


@torch.no_grad()
def scores_cache(model, cache, batch_size=64):
    # Mejor score después del NMS en cada corte del caché (0.0 si no queda ninguna caja). Misma regla que scores_por_corte.
    model.eval()
    out = []
    for i in range(0, len(cache["images"]), batch_size):
        x = torch.from_numpy(cache["images"][i:i + batch_size].astype(np.float32)).to(DEVICE)
        with autocast():
            det_logits = model.det_head(model.trunk(x)[-1])
        final = postprocess(det_logits.float(), score_thresh=CONFIG_PIPELINE["score_thresh"], nms_fn=nms_por_clase,
                            iou_thresh=CONFIG_PIPELINE["iou_thresh"], max_det_per_class=CONFIG_PIPELINE["max_det_per_class"])
        out += [float(f["scores"].max()) if f["scores"].numel() else 0.0 for f in final]
    return np.asarray(out, dtype=np.float64)


evaluate_base = evaluate
selection_score_base = selection_score
LINEA_BASE = None


def evaluate(model, loader, run_seg, instance_n=INSTANCE_EVAL_N):
    with torch.no_grad():                    # la versión original lleva @torch.no_grad(); se asegura aquí también
        out = evaluate_base(model, loader, run_seg, instance_n)
    s_pos, s_neg = scores_cache(model, val_cache), scores_cache(model, neg_val_cache)
    tiene = np.r_[np.ones(len(s_pos), bool), np.zeros(len(s_neg), bool)]
    r = cv.resumen_cortes_vacios(np.r_[s_pos, s_neg], tiene, recall_min=RECALL_MIN, umbral_fijo=UMBRAL_ACTUAL)
    out.update({f"val_{k}": v for k, v in r.items()})
    return out


def selection_score(kind, v):
    if kind != "negativos":
        return selection_score_base(kind, v)
    if LINEA_BASE is None:
        raise RuntimeError("Primero hay que medir LINEA_BASE")
    return cv.criterio_seleccion(v, LINEA_BASE)[1]


COLUMNAS_CLAVE = ["val_det_map50", "val_det_mean_iou", "val_seg_dice_pred", "val_inst_recall50_pred",
                  "val_umbral_cal", "val_recall_pelvis_cal", "val_fpr_vacios_cal", "val_fpr_vacios_fijo", "val_vacios_con_caja"]

t0 = time.time()
modelo = modelo_inicial()
LINEA_BASE = evaluate(modelo, val_loader, run_seg=True)
(OUT_ROOT / "linea_base_val.json").write_text(json.dumps({k: float(v) for k, v in LINEA_BASE.items()}, indent=2), encoding="utf-8")
print(f"Línea base medida en {time.time() - t0:.0f} s")
display(pd.Series({k: LINEA_BASE[k] for k in COLUMNAS_CLAVE}).round(4).to_frame("sin negativos"))

if not SMOKE_TEST:
    original = PAYLOAD_INICIAL["selected_from"]["val_metrics"]
    for k in ("val_det_map50", "val_seg_dice_pred"):
        d = abs(LINEA_BASE[k] - original[k])
        print(f"{k}: ahora {LINEA_BASE[k]:.4f} | en el checkpoint {original[k]:.4f} | diferencia {d:.4f}")
        if d > 0.005:
            print("   AVISO: la línea base no reproduce el checkpoint. Revisar datos, caché o dispositivo antes de seguir.")
del modelo
gc.collect()

## 7. Ajuste fino, una corrida por configuración

Cada configuración parte de **los mismos pesos** (el modelo actual) y de la misma semilla, para que la única diferencia sea la cantidad de negativos. Es la etapa C de siempre (`train_stage`, cajas guía mezcladas, AMP en CUDA), con tasa de aprendizaje 1e-4 y 4 épocas. Por época se guardan el historial, el mejor checkpoint según la regla y el último.

In [ ]:
corridas = {}
for conf in CONFIGURACIONES:
    nombre = conf["nombre"]
    OUT_DIR, MODEL_DIR = OUT_ROOT / nombre, MODEL_ROOT / nombre          # globals que usa train_stage
    OUT_DIR.mkdir(parents=True, exist_ok=True); MODEL_DIR.mkdir(parents=True, exist_ok=True)

    train_loader, n_neg = loader_entrenamiento(conf["fraccion"])
    etapa = {**ETAPA_AJUSTE, "name": f"C_negativos_{nombre}"}
    try:
        commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=ROOT, timeout=10).stdout.strip() or None
    except Exception:
        commit = None
    config = {**CONFIG_ORIGINAL, "STAGES": [etapa], "USE_AMP": USE_AMP, "SMOKE_TEST": SMOKE_TEST, "BATCH_SIZE": BATCH_SIZE,
              "VAL_BATCH_SIZE": VAL_BATCH_SIZE, "INSTANCE_EVAL_N": INSTANCE_EVAL_N, "MAX_TRAIN_SLICES": MAX_TRAIN_SLICES,
              "MAX_VAL_SLICES": MAX_VAL_SLICES, "DATA_VERSION": DATA_VERSION, "torch": torch.__version__,
              "device": NOMBRE_DISPOSITIVO, "git_commit": commit,
              "AJUSTE_NEGATIVOS": {"modelo_inicial": MODELO_INICIAL.name, "fraccion_negativos": conf["fraccion"],
                                   "n_negativos_train": n_neg, "n_positivos_train": len(train_loader.dataset.pos),
                                   "criterio": cv.CRITERIO, "umbral_corte_actual": UMBRAL_ACTUAL,
                                   "linea_base": {k: float(LINEA_BASE[k]) for k in COLUMNAS_CLAVE}}}
    (OUT_DIR / "config.json").write_text(json.dumps(config, indent=2, default=str), encoding="utf-8")

    print(f"\n##### Configuración '{nombre}': {config['AJUSTE_NEGATIVOS']['n_positivos_train']} positivos + {n_neg} negativos")
    modelo = modelo_inicial()
    t0 = time.time()
    filas = train_stage(modelo, etapa, train_loader, val_loader, config)
    historial = pd.DataFrame(filas)
    historial.to_csv(OUT_DIR / f"historial_{etapa['name']}.csv", index=False)

    mejor = historial[historial["is_best"] == True].iloc[-1].to_dict()          # train_stage marca cada mejora; la última es la mejor
    cumple, puntaje, detalle = cv.criterio_seleccion(mejor, LINEA_BASE)
    corridas[nombre] = {"conf": conf, "etapa": etapa, "historial": historial, "mejor": mejor, "cumple": cumple,
                        "puntaje": puntaje, "detalle": detalle, "checkpoint": MODEL_DIR / f"best_{etapa['name']}.pth",
                        "minutos": (time.time() - t0) / 60}
    print(f"'{nombre}': mejor época {int(mejor['epoch'])} | cumple la regla: {cumple} | {detalle} | {corridas[nombre]['minutos']:.1f} min")
    display(historial[["epoch"] + COLUMNAS_CLAVE + ["select_score"]].round(4))
    del modelo, train_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## 8. Comparación y decisión

La tabla pone la línea base junto a la mejor época de cada configuración. La decisión aplica la regla de la sección 0 sin excepciones: si ninguna configuración cumple, se conserva el modelo actual.

In [ ]:
filas = [{"modelo": "sin negativos (actual)", **{k: LINEA_BASE[k] for k in COLUMNAS_CLAVE}, "cumple": None, "epoca": None}]
for nombre, r in corridas.items():
    filas.append({"modelo": f"negativos: {nombre}", **{k: r["mejor"][k] for k in COLUMNAS_CLAVE},
                  "cumple": r["cumple"], "epoca": int(r["mejor"]["epoch"])})
comparacion = pd.DataFrame(filas)
comparacion.to_csv(OUT_ROOT / "comparacion_val.csv", index=False)
display(comparacion.round(4))

candidatas = [n for n, r in corridas.items() if r["cumple"]]
GANADOR = (max(candidatas, key=lambda n: (corridas[n]["puntaje"], corridas[n]["mejor"]["val_det_map50"]))
           if candidatas else None)
if SMOKE_TEST and GANADOR is None:
    GANADOR = CONFIGURACIONES[0]["nombre"]          # solo para recorrer las secciones 9 y 10 en la prueba corta
    print("(SMOKE_TEST) Sin ganador real: se usa la primera configuración solo para probar el resto del código.")
decision = {"ganador": GANADOR, "criterio": cv.CRITERIO, "smoke_test": SMOKE_TEST,
            "por_configuracion": {n: {"cumple": r["cumple"], "epoca": int(r["mejor"]["epoch"]), **r["detalle"]}
                                  for n, r in corridas.items()}}
(OUT_ROOT / "decision.json").write_text(json.dumps(decision, indent=2, default=float), encoding="utf-8")
if GANADOR:
    d = corridas[GANADOR]["detalle"]
    print(f"Ganador: '{GANADOR}' (época {int(corridas[GANADOR]['mejor']['epoch'])}). Cortes vacíos que pasan: "
          f"{LINEA_BASE['val_fpr_vacios_cal']:.1%} -> {d['fpr']:.1%} | caída de mAP {d['caida_map50']:+.4f} | "
          f"caída de Dice {d['caida_dice']:+.4f} | recall de pelvis {d['recall_pelvis']:.3f}")
else:
    print("Ninguna configuración cumple la regla: se conserva el modelo actual. Reportarlo como resultado negativo.")

# Curvas por época frente a la línea base
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8), constrained_layout=True)
for nombre, r in corridas.items():
    h = r["historial"]
    ax[0].plot(h["epoch"], h["val_fpr_vacios_cal"], marker="o", label=nombre)
    ax[1].plot(h["epoch"], h["val_det_map50"], marker="o", label=nombre)
    ax[2].plot(h["epoch"], h["val_seg_dice_pred"], marker="o", label=nombre)
for a, k, t in zip(ax, ("val_fpr_vacios_cal", "val_det_map50", "val_seg_dice_pred"),
                   ("Cortes vacíos que pasan (umbral recalibrado)", "mAP@0.50 (cortes positivos)", "Dice con cajas predichas")):
    a.axhline(LINEA_BASE[k], ls="--", c="gray", lw=1, label="sin negativos")
    a.set_title(t); a.set_xlabel("época"); a.legend(); a.grid(alpha=0.25)
fig.savefig(FIG_DIR / "curvas_ajuste_negativos.png", dpi=150)
plt.show()

## 9. Comprobación sobre los volúmenes completos de `val`

La sección 8 usa los cachés. Aquí se repite **exactamente** el procedimiento de `nms_pertenencia_pipeline` (§5.1): cada volumen de `val` se lee del MHA, se preprocesa corte a corte y se pasa por `scores_por_corte`. Se hace con el modelo actual y con el ganador. Para el modelo actual, los scores deben coincidir con `scores_cortes_val.csv`, lo que confirma que la comparación es justa.

También se evalúa el filtro opcional de tramo principal en z. Es informativo: solo se recomienda si quita cortes vacíos sin perder pelvis con el modelo ganador.

In [ ]:
def preprocesar_corte(corte_hu, output_size=IMAGE_SIZE):
    # Mismo preprocesamiento del dataset: ventaneo HU -> Otsu + CLAHE -> letterbox (igual que nms_pertenencia_pipeline).
    imagen = enhance_bone_contrast(apply_hu_window(corte_hu))
    imagen_256, _, transform = letterbox_pair(imagen, np.zeros(corte_hu.shape, dtype=np.int16), output_size)
    return imagen_256, transform


def componentes_de(model):
    model.eval()
    return {"backbone": model.backbone, "cbam": model.cbam, "det_head": model.det_head, "features_fn": backbone_feature_pyramid}


def scores_volumenes(model, casos_val):
    comp, filas = componentes_de(model), []
    for caso in casos_val:
        fila = df[df["case_id"] == caso].iloc[0]
        vol = sitk.GetArrayFromImage(sitk.ReadImage(str(resolve_repo_path(fila["image_path"])))).astype(np.float32)
        lab = sitk.GetArrayFromImage(sitk.ReadImage(str(resolve_repo_path(fila["label_path"]))))
        tiene = lab.reshape(lab.shape[0], -1).any(axis=1)
        for inicio in range(0, vol.shape[0], 16):
            zs = range(inicio, min(inicio + 16, vol.shape[0]))
            x = torch.from_numpy(apilar_cortes(vol, zs, lambda corte: preprocesar_corte(corte)[0], IN_CHANNELS)).to(DEVICE)
            for z, s in zip(zs, scores_por_corte(comp, x)):
                filas.append({"case_id": caso, "z": z, "tiene_hueso": bool(tiene[z]), "score_max": float(s)})
        del vol, lab
        gc.collect()
    return pd.DataFrame(filas)


CASOS_VAL = sorted(df.loc[df["split"] == "val", "case_id"])[: (1 if SMOKE_TEST else None)]
modelos_a_comparar = {"sin_negativos": None}
if GANADOR:
    modelos_a_comparar[GANADOR] = corridas[GANADOR]["checkpoint"]

volumenes = {}
for etiqueta, ckpt in modelos_a_comparar.items():
    modelo = modelo_inicial()
    if ckpt is not None:
        load_weights(modelo, ckpt)
    t0 = time.time()
    tabla = scores_volumenes(modelo, CASOS_VAL)
    tabla.to_csv(OUT_ROOT / f"scores_cortes_val_{etiqueta}.csv", index=False)
    volumenes[etiqueta] = tabla
    print(f"{etiqueta}: {len(tabla)} cortes en {time.time() - t0:.0f} s")
    del modelo
    gc.collect()

# El modelo actual debe reproducir los scores guardados por nms_pertenencia_pipeline
guardados = pd.read_csv(METRICAS_DIR / "nms_pertenencia" / "scores_cortes_val.csv", dtype={"case_id": str})
_cmp = volumenes["sin_negativos"].merge(guardados, on=["case_id", "z"], suffixes=("", "_guardado"))
print(f"Reproducción de scores_cortes_val.csv: {len(_cmp)} cortes | diferencia máxima {(_cmp['score_max'] - _cmp['score_max_guardado']).abs().max():.4f} "
      f"| mismo tiene_hueso: {bool((_cmp['tiene_hueso'] == _cmp['tiene_hueso_guardado']).all())}")

resumen_vol = pd.DataFrame({etq: cv.resumen_cortes_vacios(t["score_max"], t["tiene_hueso"], recall_min=RECALL_MIN,
                                                          umbral_fijo=UMBRAL_ACTUAL) for etq, t in volumenes.items()}).T
resumen_vol.to_csv(OUT_ROOT / "resumen_volumen_completo_val.csv")
display(resumen_vol.round(4))

fig, ax = plt.subplots(1, len(volumenes), figsize=(6.5 * len(volumenes), 3.8), constrained_layout=True, squeeze=False)
bins = np.linspace(0, 1, 41)
for a, (etq, t) in zip(ax[0], volumenes.items()):
    u = resumen_vol.loc[etq, "umbral_cal"]
    a.hist(t.loc[t["tiene_hueso"], "score_max"], bins=bins, alpha=0.7, label="con pelvis")
    a.hist(t.loc[~t["tiene_hueso"], "score_max"], bins=bins, alpha=0.7, label="sin pelvis")
    a.axvline(u, ls="--", c="k", lw=1, label=f"umbral recalibrado {u:.2f}")
    a.set_yscale("log"); a.set_xlabel("mejor score después del NMS"); a.set_title(etq); a.legend()
fig.savefig(FIG_DIR / "scores_cortes_val_comparacion.png", dpi=150)
plt.show()

for etq, t in volumenes.items():
    print(f"Filtro de tramo principal ({etq}, umbral {resumen_vol.loc[etq, 'umbral_cal']:.2f}):")
    display(cv.evaluar_tramo_principal(t, resumen_vol.loc[etq, "umbral_cal"], huecos=(0, 2, 5, 10)))

## 10. Exportar el ganador y adoptarlo

Se exporta el mejor checkpoint del ganador con el mismo formato que `modelo_multitarea_final.pt` (pesos + arquitectura) y se comprueba que el archivo reconstruido da las mismas salidas.

**Adoptar** reemplaza `resultados/modelos/entrenamiento_multitarea/modelo_multitarea_final.pt` por el modelo ajustado. Antes guarda una copia del actual como `modelo_multitarea_final_sin_negativos.pt` (solo la primera vez). Solo se puede adoptar si hay un ganador que cumple la regla y no es una prueba corta. Después de adoptar hay que volver a correr, en este orden:
1. `nms_pertenencia_pipeline`: reentrena sola la cabeza de pertenencia (detecta que el tronco cambió) y recalibra `umbral_corte`.
2. `salida_3d`: recalcula las salidas por caso con el modelo nuevo y vuelve a ajustar la asociación con `val`.
3. La evaluación final en `test`, una sola vez, con todo congelado.

In [ ]:
ADOPTAR = False            # True solo después de revisar las secciones 8 y 9

EXPORT_PATH = None
if GANADOR:
    EXPORT_PATH = MODEL_ROOT / "modelo_multitarea_negativos.pt"
    payload = export_inference_checkpoint(corridas[GANADOR]["checkpoint"], EXPORT_PATH)
    payload["ajuste_negativos"] = json.loads(json.dumps({"configuracion": corridas[GANADOR]["conf"], "decision": decision,
                                                         "resumen_volumen_val": resumen_vol.to_dict()}, default=float))
    torch.save(payload, EXPORT_PATH)
    recargado, _ = load_trained_model(EXPORT_PATH)
    referencia = modelo_inicial()
    load_weights(referencia, corridas[GANADOR]["checkpoint"])
    referencia.eval()
    x = next(iter(val_loader))["image"][:4].to(DEVICE)
    with torch.no_grad():
        assert torch.allclose(referencia.det_head(referencia.trunk(x)[-1]), recargado.det_head(recargado.trunk(x)[-1]), atol=1e-5)
    print(f"Exportado {EXPORT_PATH.relative_to(ROOT)} ({EXPORT_PATH.stat().st_size / 1e6:.1f} MB): reconstrucción OK")
    del recargado, referencia
else:
    print("No hay ganador: no se exporta nada.")

if ADOPTAR:
    if SMOKE_TEST or not GANADOR:
        raise RuntimeError("No se adopta: es una prueba corta o ninguna configuración cumplió la regla.")
    final = ORIGEN_DIR / "modelo_multitarea_final.pt"
    if not RESPALDO_SIN_NEGATIVOS.is_file():
        shutil.copy2(final, RESPALDO_SIN_NEGATIVOS)
    shutil.copy2(EXPORT_PATH, final)
    (OUT_ROOT / "adopcion.json").write_text(json.dumps({
        "fecha": time.strftime("%Y-%m-%d %H:%M"), "adoptado": EXPORT_PATH.name, "reemplaza": final.name,
        "respaldo": RESPALDO_SIN_NEGATIVOS.name, "ganador": GANADOR}, indent=2), encoding="utf-8")
    print(f"Adoptado. Respaldo del modelo anterior: {RESPALDO_SIN_NEGATIVOS.name}. "
          "Ahora correr nms_pertenencia_pipeline y salida_3d.")

## 11. Cómo leer el resultado y qué escribir en el informe

- **Qué se compara.** La misma red, los mismos pesos de partida y los mismos hiperparámetros; lo único que cambia es si el ajuste ve cortes sin pelvis. La línea base se mide con este mismo código.
- **La métrica que importa aquí** es la fracción de cortes sin pelvis que pasan el filtro (`fpr_vacios_cal`), con el umbral recalibrado para conservar al menos el 99 % de los cortes con pelvis. `fpr_vacios_fijo` usa el umbral actual (el de `umbral_corte.json`) y sirve para comparar con la misma vara.
- **Lo que no puede empeorar:** la mAP@0.50 y el Dice en cortes positivos. Si bajan más de 0,01, el ajuste no se adopta aunque reduzca los falsos positivos.
- **Limitación honesta.** El umbral y la época se eligen con los mismos 5 casos de `val`, así que el resultado en `val` es optimista. La medida definitiva es `test`, una sola vez, con todo congelado.
- **Si ninguna configuración cumple,** también es un resultado. Se reporta así: con un ajuste corto, agregar negativos no logra bajar los falsos positivos sin dañar la detección o la segmentación. Las siguientes opciones serían una cabeza de presencia por corte o un entrenamiento completo con negativos desde la etapa A.
- **Uso de IA.** Registrar en `IA_USAGE.md` qué partes de este notebook y de `src/cortes_vacios.py` salieron de un asistente de IA y qué se revisó o cambió a mano.